# 05. 머신러닝 - 회귀

> 분류와는 모델군·평가지표가 완전히 다른 트랙이라 별도 파일로 분리했습니다. 이 노트북에서는 Titanic 데이터로 **요금(Fare)을 예측하는 회귀 문제**를 다룹니다. (다른 도메인의 회귀 문제는 `notebooks_extra/EX01`에서 캘리포니아 집값 데이터로 별도 연습합니다.)

### 이 노트북에서 배우는 것
- 모델 불러오기→생성→학습→예측→평가 5단계 템플릿
- 회귀 모델(Linear/Tree/RandomForest/GradientBoosting/XGB/MLP) 실습
- mse/mae/r2 등 회귀 전용 평가지표
- feature_importances_로 변수 중요도 해석하기
- 여러 모델을 앙상블(Voting)하는 방법

### 사용 데이터
- `data/train.csv` (Titanic) - Pclass/Age/SibSp/Parch로 Fare 예측

## 목차
1. 머신러닝 모델링 템플릿 구조
2. 회귀 문제란? 모델·모듈·평가함수 총정리표
3. 회귀 모델 실습
4. 정규화 회귀(Ridge/Lasso)와 다항 회귀(PolynomialFeatures)
5. 회귀 평가지표
6. 회귀 성능 시각화
7. Feature Importance 해석
8. 여러 모델 앙상블 (VotingRegressor)
9. 종합 실전 문제

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/train.csv')
df['Age'] = df['Age'].fillna(df['Age'].median())

X = df[['Pclass', 'Age', 'SibSp', 'Parch']]
y = df['Fare']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

## 1. 머신러닝 모델링 템플릿 구조

### 📖 이론 설명
scikit-learn의 모든 모델은 거의 동일한 5단계로 사용합니다. **이 뼈대만 외워두면, 문제에서 어떤 모델을 요구하든 `[2]`단계의 클래스 이름만 바꿔 끼우면 됩니다.**

```python
# [1] 모델 불러오기
from sklearn.모듈 import 모델클래스
# [2] 모델 생성
model = 모델클래스(하이퍼파라미터=...)
# [3] 모델 학습
model.fit(X_train, y_train)
# [4] 예측
y_pred = model.predict(X_test)
# [5] 평가
평가함수(y_test, y_pred)
```

## 2. 회귀 문제란? 모델·모듈·평가함수 총정리표

### 📖 이론 설명
회귀는 '연속적인 숫자 값'을 예측하는 문제입니다(예: 요금, 집값, 온도). 분류처럼 클래스로 나누는 것이 아니라 '얼마나 가까운 값을 예측했는가'로 성능을 평가하기 때문에, 평가함수도 accuracy가 아니라 오차 기반 지표를 씁니다.

### 🧩 핵심 개념 정리
| 모델명 | 모듈 |
|---|---|
| LinearRegression | sklearn.linear_model |
| DecisionTreeRegressor | sklearn.tree |
| RandomForestRegressor | sklearn.ensemble |
| GradientBoostingRegressor | sklearn.ensemble |
| XGBRegressor | xgboost |
| MLPRegressor | sklearn.neural_network |

평가함수: `mean_squared_error`, `mean_absolute_error`, `r2_score` (모두 `sklearn.metrics`)

## 3. 회귀 모델 실습

### 📖 이론 설명
위 템플릿을 그대로 적용해 여러 회귀 모델을 학습시켜 봅니다. `y`가 pandas Series일 때 일부 모델은 `.to_numpy().flatten()`으로 1차원 배열로 바꿔줘야 경고 없이 동작하는 경우가 있으니 알아두면 좋습니다.

### 💻 예제 코드

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

xgb = XGBRegressor(n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

### ✍️ TODO: 실전 문제

**문제 1.** `GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)`를 학습시키고 예측값 `gb_pred`를 만드세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.ensemble import GradientBoostingRegressor
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)  # learning_rate: 부스팅 각 단계의 반영 비율(작을수록 안정적이지만 트리가 더 많이 필요)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
```

</details>

**문제 2.** `DecisionTreeRegressor(max_depth=5, random_state=42)`를 학습시키고 예측값 `dt_pred`를 만드세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.tree import DecisionTreeRegressor
dt = DecisionTreeRegressor(max_depth=5, random_state=42)  # max_depth로 깊이를 제한하지 않으면 훈련 데이터에 과적합되기 쉬움
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
```

</details>

**문제 3.** `XGBRegressor(n_estimators=300, early_stopping_rounds=10, random_state=42)`를 `eval_set=[(X_test, y_test)]`로 학습시켜, 실제로 몇 번째 트리에서 멈췄는지(`best_iteration`) 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
xgb_es = XGBRegressor(n_estimators=300, early_stopping_rounds=10, random_state=42)  # early_stopping_rounds는 최신 XGBoost에서 fit()이 아니라 생성자 파라미터로 지정
xgb_es.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)  # eval_set의 성능이 10라운드 연속 개선되지 않으면 n_estimators(300)를 다 채우지 않고 조기 종료
print('실제 사용된 트리 개수:', xgb_es.best_iteration)
xgb_es_pred = xgb_es.predict(X_test)
```

</details>

## 4. 정규화 회귀(Ridge/Lasso)와 다항 회귀(PolynomialFeatures)

### 📖 이론 설명
`LinearRegression`은 계수 크기에 아무 제약이 없어서, 변수가 많거나 변수끼리 상관관계가 강하면 (다중공선성) 계수가 불안정해지거나 과적합되기 쉽습니다. **Ridge(L2 정규화)**와 **Lasso(L1 정규화)**는 계수 크기 자체에 페널티를 주어 이를 억제합니다. 특히 Lasso는 중요하지 않은 변수의 계수를 **정확히 0**으로 만들어버려서 자동으로 변수를 선택하는 효과도 있습니다. `PolynomialFeatures`는 기존 변수들의 거듭제곱·곱셈 조합을 새 변수로 추가해, 선형 모델로도 비선형적인 관계를 어느 정도 잡아낼 수 있게 해줍니다(다만 변수가 급격히 늘어나 과적합 위험도 커지므로 Ridge/Lasso와 함께 쓰는 경우가 많습니다).

### 🧩 핵심 개념 정리
| 함수 | 설명 |
|---|---|
| Ridge(alpha=) | L2 정규화 회귀. alpha가 클수록 계수를 더 강하게 억제 |
| Lasso(alpha=) | L1 정규화 회귀. 일부 계수를 정확히 0으로 만들어 변수 선택 효과 |
| PolynomialFeatures(degree=) | 기존 변수의 거듭제곱·곱 조합을 새 변수로 추가 |

### 💻 예제 코드

In [ ]:
from sklearn.linear_model import Ridge, Lasso

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)

lasso = Lasso(alpha=1.0)
lasso.fit(X_train, y_train)
lasso_pred = lasso.predict(X_test)

print('LinearRegression 계수:', lr.coef_)
print('Lasso 계수:', lasso.coef_)  # 일부 계수가 0에 가깝거나 정확히 0인지 확인

### ✍️ TODO: 실전 문제

**문제 1.** `PolynomialFeatures(degree=2)`로 `X_train`, `X_test`를 변환한 `X_train_poly`, `X_test_poly`를 만들고, 이 데이터로 `LinearRegression`을 학습시켜 `poly_pred`를 만드세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

poly_lr = LinearRegression()
poly_lr.fit(X_train_poly, y_train)
poly_pred = poly_lr.predict(X_test_poly)
print(X_train_poly.shape)  # 원래 4개 변수보다 컬럼 수가 늘어남(거듭제곱/곱 조합이 추가됨)
```

</details>

**문제 2.** `Ridge(alpha=10.0)`으로 `ridge_strong`을 학습시키고, `alpha=1.0`일 때(`ridge`)와 계수 크기를 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
ridge_strong = Ridge(alpha=10.0)
ridge_strong.fit(X_train, y_train)
print('alpha=1 :', ridge.coef_)
print('alpha=10:', ridge_strong.coef_)  # alpha가 클수록 계수가 더 0에 가깝게 눌림(정규화가 더 강해짐)
```

</details>

## 5. 회귀 평가지표

### 📖 이론 설명
- **MSE(평균제곱오차)**: 오차를 제곱해서 평균 - 큰 오차에 더 민감(패널티가 큼)
- **MAE(평균절대오차)**: 오차의 절댓값 평균 - MSE보다 이상치에 덜 민감
- **R²(결정계수)**: 모델이 데이터의 분산을 얼마나 설명하는지, 1에 가까울수록 좋음(0이면 평균만 예측한 것과 비슷, 음수면 평균보다도 못함)

### 🧩 핵심 개념 정리
| 함수 | 낮을수록 좋음/높을수록 좋음 |
|---|---|
| mean_squared_error | 낮을수록 좋음 |
| mean_absolute_error | 낮을수록 좋음 |
| r2_score | 높을수록 좋음(최대 1) |

### 💻 예제 코드

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

for name, pred in [('LinearRegression', lr_pred), ('RandomForest', rf_pred), ('XGBoost', xgb_pred)]:
    mse = mean_squared_error(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    print(f'{name}: MSE={mse:.2f}, MAE={mae:.2f}, R2={r2:.4f}')

### ✍️ TODO: 실전 문제

**문제 1.** `rf_pred`(RandomForest 예측값)의 RMSE(=MSE의 제곱근)를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
rmse = mean_squared_error(y_test, rf_pred) ** 0.5  # RMSE = MSE의 제곱근: 오차의 단위를 원래 값과 동일하게 되돌려 해석이 쉬움
print(rmse)
```

</details>

## 6. 회귀 성능 시각화

### 📖 이론 설명
예측값과 실제값이 얼마나 가까운지는 산점도로, 오차(잔차 = 실제값-예측값)의 분포는 잔차 플롯으로 확인합니다. 산점도에서 점들이 대각선(y=x)에 가까울수록 예측이 잘 맞은 것이고, 잔차 플롯에서 0을 기준으로 점들이 골고루 퍼져 있으면 편향이 없다는 뜻입니다.

### 🧩 핵심 개념 정리
| 그래프 | 해석 |
|---|---|
| 예측 vs 실제 산점도 | 대각선에 가까울수록 좋은 예측 |
| 잔차 플롯 (residual) | 0 근처에 무작위로 퍼져 있어야 편향 없음 |

### 💻 예제 코드

In [ ]:
plt.scatter(y_test, rf_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Fare')
plt.ylabel('Predicted Fare')
plt.title('RandomForest: Actual vs Predicted')
plt.show()

residuals = y_test - rf_pred
plt.scatter(rf_pred, residuals, alpha=0.5)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predicted Fare')
plt.ylabel('Residual')
plt.title('Residual Plot')
plt.show()

### ✍️ TODO: 실전 문제

**문제 1.** `xgb_pred`에 대한 예측 vs 실제 산점도를 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
plt.scatter(y_test, xgb_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')  # y=x 대각선을 함께 그려 점들이 이 선에 가까울수록 예측이 정확하다는 것을 시각적으로 보여줌
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('XGBoost: Actual vs Predicted')
plt.show()
```

</details>

## 7. Feature Importance 해석

### 📖 이론 설명
트리 기반 모델(DecisionTree, RandomForest, XGBoost 등)은 학습 후 `model.feature_importances_` 속성으로 각 변수가 예측에 얼마나 기여했는지 알 수 있습니다. '가장 중요한 변수가 무엇인가'를 묻는 문제에서 바로 활용합니다.

### 💻 예제 코드

In [ ]:
importances = rf.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)
print(feat_imp)

plt.barh(feat_imp.index, feat_imp.values)
plt.title('RandomForest Feature Importance')
plt.show()

### ✍️ TODO: 실전 문제

**문제 1.** `xgb` 모델의 feature importance를 구해 가장 중요한 변수 이름을 `top_feature`에 저장하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
xgb_imp = pd.Series(xgb.feature_importances_, index=X.columns)  # feature_importances_ 배열에 컬럼명을 인덱스로 붙여 어떤 값이 어떤 변수인지 알 수 있게 함
top_feature = xgb_imp.idxmax()  # idxmax(): 값이 아니라 값이 가장 큰 위치(컬럼명)를 반환
print(top_feature)
```

</details>

## 8. 여러 모델 앙상블 (VotingRegressor)

### 📖 이론 설명
서로 다른 모델들의 예측을 평균 내면, 개별 모델의 약점이 상쇄되어 더 안정적인 예측을 얻을 수 있는 경우가 많습니다. `VotingRegressor`는 `(이름, 모델)` 튜플의 리스트를 받아 여러 모델을 하나처럼 다룹니다. 학습이 끝난 모델은 `joblib.dump`로 저장해뒀다가 나중에 `joblib.load`로 다시 불러와 재사용할 수 있습니다.

### 💻 예제 코드

In [ ]:
from sklearn.ensemble import VotingRegressor
import joblib

voting_models = [('linear', LinearRegression()), ('rf', RandomForestRegressor(n_estimators=50, random_state=42))]
voting = VotingRegressor(voting_models)
voting.fit(X_train, y_train)
voting_pred = voting.predict(X_test)
print('Voting R2:', r2_score(y_test, voting_pred))

# 모델 저장 & 불러오기
joblib.dump(voting, 'voting_model.pkl')
loaded_model = joblib.load('voting_model.pkl')
print(loaded_model.predict(X_test.iloc[:3]))

## 9. 종합 실전 문제

**회귀 모델링 전체 흐름을 이어서 풀어보는 미니 모의고사입니다.**

**문제 1.** `RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)`를 학습시키고 R2 점수를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
rf2 = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)  # n_estimators(트리 개수)를 늘리면 보통 안정적이지만 학습 시간도 늘어남
rf2.fit(X_train, y_train)
print(r2_score(y_test, rf2.predict(X_test)))
```

</details>

**문제 2.** `LinearRegression`과 `XGBRegressor`, `RandomForestRegressor`를 `VotingRegressor`로 묶어 학습시키고 MAE를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
voting2 = VotingRegressor([('lr', LinearRegression()), ('xgb', XGBRegressor(random_state=42)), ('rf', RandomForestRegressor(random_state=42))])  # 서로 다른 성격의 모델(선형/부스팅/배깅)을 섞으면 개별 약점이 상쇄되는 경우가 많음
voting2.fit(X_train, y_train)
print(mean_absolute_error(y_test, voting2.predict(X_test)))
```

</details>